### Convert all CSVS to parquet

In [ ]:
# Save aggregates to parquet and display simple plots
import os
import pandas as pd
import matplotlib.pyplot as plt

DEST_DIR = r"C:\Users\Matt\Desktop\CS506\CS506_Project\1_LIB\nyiso\nyiso_parquet"
os.makedirs(DEST_DIR, exist_ok=True)

# Ensure aggregates exist; if not, (re)load from parquet files defined in previous cell
try:
    hourly
except NameError:
    print('hourly not found in kernel namespace — loading parquets and building aggregates')
    paths = find_parquets(SOURCE_DIR)
    df_all = load_parquets(paths)
    df_all = ensure_datetime(df_all)
    df_all.set_index('Time Stamp', inplace=True)
    hourly = df_all['Load'].resample('H').sum().to_frame('total_load')
    daily = df_all['Load'].resample('D').sum().to_frame('total_load')
    per_name_daily = df_all.groupby([pd.Grouper(freq='D'), 'Name'])['Load'].sum().unstack(fill_value=0)

# Paths to write aggregates
hourly_path = os.path.join(DEST_DIR, '2023_hourly_total_load.parquet')
daily_path = os.path.join(DEST_DIR, '2023_daily_total_load.parquet')
per_name_path = os.path.join(DEST_DIR, '2023_per_name_daily.parquet')

# Write aggregates to parquet (overwrites if present)
hourly.to_parquet(hourly_path)
daily.to_parquet(daily_path)
per_name_daily.to_parquet(per_name_path)
print(f'Saved hourly -> {hourly_path}')
print(f'Saved daily -> {daily_path}')
print(f'Saved per-name daily -> {per_name_path}')

# Inline quick plots
plt.figure(figsize=(10,4))
daily['total_load'].plot(title='Daily Total Load (2023)')
plt.ylabel('Total Load')
plt.tight_layout()
plt.show()

plt.figure(figsize=(10,5))
try:
    top10 = per_name_daily.mean().sort_values(ascending=False).head(10)
    top10.plot(kind='bar')
    plt.title('Top 10 Names by Avg Daily Load (2023)')
    plt.ylabel('Avg Daily Load')
    plt.tight_layout()
    plt.show()
except Exception as e:
    print('Could not plot top10 per-name:', e)

In [ ]:
"""Convert CSV files to Parquet preserving directory structure.

Usage:
    python scripts/convert_csvs_to_parquet.py \
        --source "C:\\path\\to\\nyiso_csvs" \
        --dest   "C:\\path\\to\\nyiso_parquet"

The script will walk the source folder recursively, convert each .csv to .parquet
source
import os, glob
import pandas as pd
import numpy as np

# Source folder containing 2023 parquet files
SOURCE_DIR = r"C:\Users\Matt\Desktop\CS506\CS506_Project\1_LIB\nyiso\nyiso_parquet\2023"

def find_parquets(src):
    pattern = os.path.join(src, '**', '*.parquet')
    return sorted(glob.glob(pattern, recursive=True))

def load_parquets(paths):
    dfs = []
    for p in paths:
        try:
            df = pd.read_parquet(p)
            dfs.append(df)
        except Exception as e:
            print(f"Warning: failed to read {p}: {e}")
    if not dfs:
        return pd.DataFrame()
    return pd.concat(dfs, ignore_index=True)

def ensure_datetime(df, col='Time Stamp'):
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')
    else:
        for alt in ['Timestamp','timestamp','time','datetime','time_stamp']:
            if alt in df.columns:
                df.rename(columns={alt: col}, inplace=True)
                df[col] = pd.to_datetime(df[col], errors='coerce')
                break
    return df

# --- Run loading and aggregation ---
paths = find_parquets(SOURCE_DIR)
print(f"Found {len(paths)} parquet files under {SOURCE_DIR}")

df_all = load_parquets(paths)
print(f"Loaded rows: {len(df_all)}")

if df_all.empty:
    print('No data loaded from parquet files')
else:
    df_all = ensure_datetime(df_all)
    print('Columns present:', df_all.columns.tolist())
    if 'Load' in df_all.columns:
        print('Load statistics:')
        print(df_all['Load'].describe())
    # Create time-based aggregates in memory
    if 'Time Stamp' in df_all.columns and 'Load' in df_all.columns:
        df_all.set_index('Time Stamp', inplace=True)
        hourly = df_all['Load'].resample('H').sum().to_frame('total_load')
        daily = df_all['Load'].resample('D').sum().to_frame('total_load')
        # also per-name daily totals (wide form)
        per_name_daily = df_all.groupby([pd.Grouper(freq='D'), 'Name'])['Load'].sum().unstack(fill_value=0)
        print('Created aggregates in-memory: variables `hourly`, `daily`, `per_name_daily`')
        # show small samples
        display(hourly.head())
        display(daily.head())
    else:
        print('Missing Time Stamp or Load column; cannot aggregate')

# End of cell

usage: ipykernel_launcher.py [-h] --source SOURCE --dest DEST
                             [--engine ENGINE]
ipykernel_launcher.py: error: the following arguments are required: --source, --dest


SystemExit: 2

c:\ProgramData\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


### Summary Statistics

In [6]:
# Read and aggregate all 2023 parquet files into memory
import os, glob
import pandas as pd
import numpy as np

# SOURCE_DIR already defined in an earlier cell; redefine here for safety if needed
SOURCE_DIR = r"C:\Users\Matt\Desktop\CS506\CS506_Project\1_LIB\nyiso\nyiso_parquet\2023"

def find_parquets(src):
    pattern = os.path.join(src, '**', '*.parquet')
    return sorted(glob.glob(pattern, recursive=True))

def load_parquets(paths):
    dfs = []
    for p in paths:
        try:
            df = pd.read_parquet(p)
            df['_source_file'] = os.path.basename(p)
            dfs.append(df)
        except Exception as e:
            print(f"Warning: failed to read {p}: {e}")
    if not dfs:
        return pd.DataFrame()
    return pd.concat(dfs, ignore_index=True)

def ensure_datetime(df, col='Time Stamp'):
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')
    else:
        for alt in ['Timestamp','timestamp','time','datetime','time_stamp']:
            if alt in df.columns:
                df.rename(columns={alt: col}, inplace=True)
                df[col] = pd.to_datetime(df[col], errors='coerce')
                break
    return df

# --- Load files into memory ---
paths = find_parquets(SOURCE_DIR)
print(f"Found {len(paths)} parquet files under {SOURCE_DIR}")

df_all = load_parquets(paths)
print(f"Loaded total rows: {len(df_all)}")

# Keep dataframe in memory as `df_all` and build aggregates
if df_all.empty:
    print('No data loaded from parquet files')
else:
    df_all = ensure_datetime(df_all)
    # basic summary
    print('Columns:', df_all.columns.tolist())
    if 'Load' in df_all.columns:
        print(df_all['Load'].describe())
    # Create time-based aggregates in-memory
    if 'Time Stamp' in df_all.columns and 'Load' in df_all.columns:
        df_all.set_index('Time Stamp', inplace=True)
        hourly = df_all['Load'].resample('H').sum().to_frame('total_load')
        daily = df_all['Load'].resample('D').sum().to_frame('total_load')
        per_name_daily = df_all.groupby([pd.Grouper(freq='D'), 'Name'])['Load'].sum().unstack(fill_value=0)
        print('Aggregates created in memory: `hourly`, `daily`, `per_name_daily`')
    else:
        print('Missing Time Stamp or Load column; cannot create aggregates')

# At this point the variables `df_all`, `hourly`, `daily`, `per_name_daily` are stored in memory for further use


Found 365 parquet files under C:\Users\Matt\Desktop\CS506\CS506_Project\1_LIB\nyiso\nyiso_parquet\2023
Loaded total rows: 1180025
Columns: ['Time Stamp', 'Time Zone', 'Name', 'PTID', 'Load', '_source_file']
count    1.179992e+06
mean     1.527415e+03
std      1.429003e+03
min      1.265235e+02
25%      6.872794e+02
50%      1.093257e+03
75%      1.695525e+03
max      1.042956e+04
Name: Load, dtype: float64
Loaded total rows: 1180025
Columns: ['Time Stamp', 'Time Zone', 'Name', 'PTID', 'Load', '_source_file']
count    1.179992e+06
mean     1.527415e+03
std      1.429003e+03
min      1.265235e+02
25%      6.872794e+02
50%      1.093257e+03
75%      1.695525e+03
max      1.042956e+04
Name: Load, dtype: float64


C:\Users\Matt\AppData\Local\Temp\ipykernel_6752\113836314.py:56: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hourly = df_all['Load'].resample('H').sum().to_frame('total_load')


Aggregates created in memory: `hourly`, `daily`, `per_name_daily`


### Graph


In [ ]:
nyiso_15 = per_name_daily 

# keep only nyiso_15 (leave dunder names alone)
_keep = {'nyiso_15'}
for _name in list(globals().keys()):
    if _name in _keep or _name.startswith('__'):
        continue
    try:
        del globals()[_name]
    except Exception:
        pass

print('Kept variables:', [n for n in globals() if n in _keep])

In [ ]:
from pathlib import Path
import subprocess
import math
import sys

# scripts/git_batch_push_mesonet.py
# Commit and push files under 1_LIB/mesonet in batches.
# Usage: run this in the repo (or from notebook cell) where .git lives.


# Configuration
TARGET_DIR = Path("1_LIB/mesonet")   # relative to repo root / current working dir
BATCH_SIZE = 50                      # files per commit
PUSH_AFTER_EACH_COMMIT = True        # set False to push once at the end
DRY_RUN = False                      # set True to preview batches without running git

def run_cmd(cmd):
    subprocess.run(cmd, check=True)

def find_repo_root():
    p = subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True)
    if p.returncode != 0:
        raise RuntimeError("Not inside a git repository (git rev-parse failed).")
    return Path(p.stdout.strip())

def chunked(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

def main():
    repo_root = find_repo_root()
    target = (repo_root / TARGET_DIR).resolve()
    if not target.exists() or not target.is_dir():
        print(f"Target directory not found: {target}")
        sys.exit(1)

    files = [p.relative_to(repo_root).as_posix() for p in target.rglob("*") if p.is_file()]
    if not files:
        print("No files found under", target)
        return

    batches = list(chunked(files, BATCH_SIZE))
    print(f"Repo root: {repo_root}")
    print(f"Found {len(files)} files under {TARGET_DIR}; will create {len(batches)} commit(s) (batch size {BATCH_SIZE}).")

    if DRY_RUN:
        for i, b in enumerate(batches, start=1):
            print(f"\nBatch {i}/{len(batches)} - {len(b)} files:")
            for f in b:
                print("  ", f)
        return

    confirm = input("Proceed with staging/committing/pushing these batches? [y/N]: ").strip().lower()
    if confirm != "y":
        print("Aborted by user.")
        return

    for i, batch in enumerate(batches, start=1):
        print(f"\nProcessing batch {i}/{len(batches)} ({len(batch)} files)...")
        try:
            run_cmd(["git", "add", "--"] + batch)
            msg = f"mesonet: add batch {i}/{len(batches)} ({len(batch)} files)"
            run_cmd(["git", "commit", "-m", msg])
            if PUSH_AFTER_EACH_COMMIT:
                print("Pushing...")
                run_cmd(["git", "push"])
        except subprocess.CalledProcessError as e:
            print("Git command failed:", e)
            print("Stopping further batches.")
            break

    if not PUSH_AFTER_EACH_COMMIT:
        print("Pushing all commits...")
        run_cmd(["git", "push"])

    print("Done.")

if __name__ == "__main__":
    main()

In [ ]:

# Make sure git commands run from the repository root so relative pathspecs match.
p = subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True)
if p.returncode == 0:
    repo_root = p.stdout.strip()
    try:
        os.chdir(repo_root)
        print(f"Changed working directory to repo root: {repo_root}")
    except Exception as e:
        print("Failed to chdir to repo root:", e)
else:
    print("Could not determine git repo root; git commands will run from", os.getcwd())

from pathlib import Path
import subprocess
import math
import sys
import os

# scripts/git_batch_push_mesonet.py
# Commit and push files under 1_LIB/mesonet in batches.
# Usage: run this in the repo (or from notebook cell) where .git lives.


# Configuration
TARGET_DIR = Path("1_LIB/mesonet")   # relative to repo root / current working dir
BATCH_SIZE = 50                      # files per commit
PUSH_AFTER_EACH_COMMIT = True        # set False to push once at the end
DRY_RUN = False                      # set True to preview batches without running git

def run_cmd(cmd):
    # Run command and show output; on failure, try fallback for large 'git add' batches.
    try:
        p = subprocess.run(cmd, check=True, capture_output=True, text=True)
        if p.stdout:
            print(p.stdout.strip())
        return p
    except subprocess.CalledProcessError as e:
        stderr = e.stderr.strip() if e.stderr else str(e)
        print(f"Command failed: {cmd}\n{stderr}")
        # Fallback: if git add failed for a batch, try adding files individually to find/skips bad ones.
        if len(cmd) >= 3 and cmd[0] == "git" and cmd[1] == "add":
            # Extract file list (allow for optional "--")
            files = cmd[2:]
            if files and files[0] == "--":
                files = files[1:]
            failed = []
            for f in files:
                try:
                    subprocess.run(["git", "add", "--", f], check=True, capture_output=True, text=True)
                except subprocess.CalledProcessError as e2:
                    err2 = e2.stderr.strip() if e2.stderr else str(e2)
                    print(f"git add failed for {f}: {err2}")
                    failed.append(f)
            if failed:
                print(f"Skipped {len(failed)} file(s) due to git add failures.")
            else:
                print("All files in batch added individually after batch add failed.")
            return None
        # For other git failures, re-raise so caller can handle/stop
        raise
    subprocess.run(cmd, check=True)

def find_repo_root():
    p = subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True)
    if p.returncode != 0:
        raise RuntimeError("Not inside a git repository (git rev-parse failed).")
    return Path(p.stdout.strip())

def chunked(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

def main():
    repo_root = find_repo_root()
    target = (repo_root / TARGET_DIR).resolve()
    if not target.exists() or not target.is_dir():
        print(f"Target directory not found: {target}")
        sys.exit(1)

    files = [p.relative_to(repo_root).as_posix() for p in target.rglob("*") if p.is_file()]
    if not files:
        print("No files found under", target)
        return

    batches = list(chunked(files, BATCH_SIZE))
    print(f"Repo root: {repo_root}")
    print(f"Found {len(files)} files under {TARGET_DIR}; will create {len(batches)} commit(s) (batch size {BATCH_SIZE}).")

    if DRY_RUN:
        for i, b in enumerate(batches, start=1):
            print(f"\nBatch {i}/{len(batches)} - {len(b)} files:")
            for f in b:
                print("  ", f)
        return

    confirm = input("Proceed with staging/committing/pushing these batches? [y/N]: ").strip().lower()
    if confirm != "y":
        print("Aborted by user.")
        return

    for i, batch in enumerate(batches, start=1):
        print(f"\nProcessing batch {i}/{len(batches)} ({len(batch)} files)...")
        try:
            run_cmd(["git", "add", "--"] + batch)
            msg = f"mesonet: add batch {i}/{len(batches)} ({len(batch)} files)"
            run_cmd(["git", "commit", "-m", msg])
            if PUSH_AFTER_EACH_COMMIT:
                print("Pushing...")
                run_cmd(["git", "push"])
        except subprocess.CalledProcessError as e:
            print("Git command failed:", e)
            print("Stopping further batches.")
            break

    if not PUSH_AFTER_EACH_COMMIT:
        print("Pushing all commits...")
        run_cmd(["git", "push"])

    print("Done.")

if __name__ == "__main__":
    main()

Repo root: C:\Users\Matt\Desktop\CS506\CS506_Project
Found 3705 files under 1_LIB\mesonet; will create 75 commit(s) (batch size 50).

Processing batch 1/75 (50 files)...
Command failed: ['git', 'add', '--', '1_LIB/mesonet/20150810.csv', '1_LIB/mesonet/20150811.csv', '1_LIB/mesonet/20150812.csv', '1_LIB/mesonet/20150813.csv', '1_LIB/mesonet/20150814.csv', '1_LIB/mesonet/20150815.csv', '1_LIB/mesonet/20150816.csv', '1_LIB/mesonet/20150817.csv', '1_LIB/mesonet/20150818.csv', '1_LIB/mesonet/20150819.csv', '1_LIB/mesonet/20150820.csv', '1_LIB/mesonet/20150821.csv', '1_LIB/mesonet/20150822.csv', '1_LIB/mesonet/20150823.csv', '1_LIB/mesonet/20150824.csv', '1_LIB/mesonet/20150825.csv', '1_LIB/mesonet/20150826.csv', '1_LIB/mesonet/20150827.csv', '1_LIB/mesonet/20150828.csv', '1_LIB/mesonet/20150829.csv', '1_LIB/mesonet/20150830.csv', '1_LIB/mesonet/20150831.csv', '1_LIB/mesonet/20150901.csv', '1_LIB/mesonet/20150902.csv', '1_LIB/mesonet/20150903.csv', '1_LIB/mesonet/20150904.csv', '1_LIB/mesone